In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

import os
import joblib
import json

model_dir = "/content/drive/MyDrive/Emotion_Recognition/model"

print("Files in model folder:")
print(os.listdir(model_dir))

model = joblib.load(
    os.path.join(model_dir, "emotion_xgboost.joblib")
)

encoder = joblib.load(
    os.path.join(model_dir, "label_encoder.joblib")
)

with open(os.path.join(model_dir, "metadata.json"), "r") as f:
    metadata = json.load(f)

print("\nExpected features:", model.n_features_in_)
print("Emotion labels:", encoder.classes_)
print("\nModel metadata:")
print(metadata)


Files in model folder:
['emotion_xgboost.joblib', 'label_encoder.joblib', 'metadata.json']

Expected features: 360
Emotion labels: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']

Model metadata:
{'model': 'XGBoost', 'features': 360, 'accuracy_percent': 68.06, 'classes': ['angry', 'calm', 'disgust', 'fearful', 'happy', 'neutral', 'sad', 'surprised'], 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}


In [ ]:

!pip install librosa soundfile -q


In [ ]:

import os
import librosa
import numpy as np

def extract_improved_features(file_path):
    audio, sr = librosa.load(
        file_path,
        sr=22050,
        mono=True
    )

    mfcc = librosa.feature.mfcc(
        y=audio, sr=sr, n_mfcc=40
    )

    chroma = librosa.feature.chroma_stft(
        y=audio, sr=sr, n_chroma=12
    )

    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=128
    )

    features = np.concatenate([
        np.mean(mfcc, axis=1),
        np.std(mfcc, axis=1),
        np.mean(chroma, axis=1),
        np.std(chroma, axis=1),
        np.mean(mel, axis=1),
        np.std(mel, axis=1)
    ])

    return features

audio_dir = "/content/drive/MyDrive/Emotion_Recognition"

print("Drive folder contents:")
print(os.listdir(audio_dir))


Drive folder contents:
['Audio_Speech_Actors_01-24.zip', 'processed_data', 'ravdess', 'model']


In [ ]:

import os

audio_root = "/content/drive/MyDrive/Emotion_Recognition/ravdess"

for root, dirs, files in os.walk(audio_root):
    wav_files = [f for f in files if f.lower().endswith(".wav")]

    if wav_files:
        print("Folder:", root)
        print("Example audio:", wav_files[0])
        print("Total WAV files in this folder:", len(wav_files))
        break


Folder: /content/drive/MyDrive/Emotion_Recognition/ravdess/Actor_01
Example audio: 03-01-03-01-02-01-01.wav
Total WAV files in this folder: 60


In [ ]:

audio_path = (
    "/content/drive/MyDrive/Emotion_Recognition/"
    "ravdess/Actor_01/03-01-03-01-02-01-01.wav"
)


features = extract_improved_features(audio_path)

print("Feature vector shape:", features.shape)


features = features.reshape(1, -1)


predicted_class = model.predict(features)[0]
predicted_emotion = encoder.inverse_transform(
    [predicted_class]
)[0]


probabilities = model.predict_proba(features)[0]

print("\nPredicted emotion:", predicted_emotion)
print("\nEmotion probabilities:")

for emotion, probability in zip(
    encoder.classes_, probabilities
):
    print(f"{emotion}: {probability * 100:.2f}%")


Feature vector shape: (360,)

Predicted emotion: happy

Emotion probabilities:
angry: 0.12%
calm: 0.95%
disgust: 0.65%
fearful: 0.34%
happy: 94.45%
neutral: 0.53%
sad: 1.34%
surprised: 1.62%


In [ ]:

!pip install streamlit pyngrok -q


In [ ]:

%%writefile /content/drive/MyDrive/Emotion_Recognition/app.py
import os
import joblib
import librosa
import numpy as np
import streamlit as st
import matplotlib.pyplot as plt

# ---------- Paths ----------
BASE_DIR = "/content/drive/MyDrive/Emotion_Recognition"
MODEL_DIR = os.path.join(BASE_DIR, "model")

# ---------- Page setup ----------
st.set_page_config(
    page_title="Speech Emotion Recognition",
    page_icon="🎙️",
    layout="centered"
)

# ---------- Load saved model ----------
@st.cache_resource
def load_model():
    model = joblib.load(
        os.path.join(MODEL_DIR, "emotion_xgboost.joblib")
    )
    encoder = joblib.load(
        os.path.join(MODEL_DIR, "label_encoder.joblib")
    )
    return model, encoder

model, encoder = load_model()

# ---------- Feature extraction ----------
def extract_improved_features(file_path):
    audio, sr = librosa.load(
        file_path,
        sr=22050,
        mono=True
    )

    mfcc = librosa.feature.mfcc(
        y=audio, sr=sr, n_mfcc=40
    )

    chroma = librosa.feature.chroma_stft(
        y=audio, sr=sr, n_chroma=12
    )

    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=128
    )

    features = np.concatenate([
        np.mean(mfcc, axis=1),
        np.std(mfcc, axis=1),
        np.mean(chroma, axis=1),
        np.std(chroma, axis=1),
        np.mean(mel, axis=1),
        np.std(mel, axis=1)
    ])

    return features

# ---------- Interface ----------
st.title("🎙️ Speech Emotion Recognition")
st.write(
    "Upload a speech recording and let the AI model "
    "estimate the emotion expressed in the voice."
)

st.info(
    "Supported emotions: Angry, Calm, Disgust, Fearful, "
    "Happy, Neutral, Sad and Surprised."
)

uploaded_file = st.file_uploader(
    "Upload an audio recording",
    type=["wav"]
)

if uploaded_file is not None:
    st.audio(uploaded_file)

    if st.button("Analyze Emotion", type="primary"):
        temp_path = os.path.join(
            "/tmp", uploaded_file.name
        )

        try:
            with open(temp_path, "wb") as f:
                f.write(uploaded_file.getvalue())

            with st.spinner("Analyzing your audio..."):
                features = extract_improved_features(temp_path)

                if features.shape[0] != model.n_features_in_:
                    st.error("The extracted feature count does not match the model.")
                    st.stop()

                features = features.reshape(1, -1)

                prediction = model.predict(features)[0]
                emotion = encoder.inverse_transform(
                    [prediction]
                )[0]

                probabilities = model.predict_proba(features)[0]

            st.success("Analysis complete!")

            st.subheader("Predicted Emotion")
            st.markdown(f"# 🎭 {emotion.capitalize()}")

            st.subheader("Emotion Confidence Scores")

            fig, ax = plt.subplots(figsize=(8, 4))
            ax.barh(
                encoder.classes_,
                probabilities * 100
            )
            ax.set_xlabel("Model confidence (%)")
            ax.set_xlim(0, 100)
            ax.invert_yaxis()
            fig.tight_layout()

            st.pyplot(fig)
            plt.close(fig)

            for label, probability in sorted(
                zip(encoder.classes_, probabilities),
                key=lambda item: item[1],
                reverse=True
            ):
                st.write(
                    f"**{label.capitalize()}**: "
                    f"{probability * 100:.2f}%"
                )

            st.caption(
                "Predictions are estimates from the model, "
                "not definitive judgments of a person's feelings."
            )

        except Exception as e:
            st.error(f"Could not analyze this audio: {e}")

        finally:
            if os.path.exists(temp_path):
                os.remove(temp_path)


Writing /content/drive/MyDrive/Emotion_Recognition/app.py


In [ ]:

import os

app_path = (
    "/content/drive/MyDrive/"
    "Emotion_Recognition/app.py"
)

print("App file exists:", os.path.exists(app_path))

if os.path.exists(app_path):
    print("File size:", os.path.getsize(app_path), "bytes")


App file exists: True
File size: 3902 bytes


In [ ]:

%%writefile /content/drive/MyDrive/Emotion_Recognition/app.py
import os
import tempfile
import joblib
import librosa
import numpy as np
import streamlit as st
import matplotlib.pyplot as plt

# ---------- Paths ----------
BASE_DIR = "/content/drive/MyDrive/Emotion_Recognition"
MODEL_DIR = os.path.join(BASE_DIR, "model")

# ---------- Page setup ----------
st.set_page_config(
    page_title="Speech Emotion Recognition",
    page_icon="🎙️",
    layout="centered"
)

# ---------- Load saved model ----------
@st.cache_resource
def load_model():
    model = joblib.load(
        os.path.join(MODEL_DIR, "emotion_xgboost.joblib")
    )
    encoder = joblib.load(
        os.path.join(MODEL_DIR, "label_encoder.joblib")
    )
    return model, encoder


model, encoder = load_model()

# ---------- Feature extraction ----------
# Keep this identical to the feature extraction used during training.
def extract_improved_features(file_path):
    audio, sr = librosa.load(
        file_path,
        sr=22050,
        mono=True
    )

    mfcc = librosa.feature.mfcc(
        y=audio, sr=sr, n_mfcc=40
    )

    chroma = librosa.feature.chroma_stft(
        y=audio, sr=sr, n_chroma=12
    )

    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=128
    )

    features = np.concatenate([
        np.mean(mfcc, axis=1),
        np.std(mfcc, axis=1),
        np.mean(chroma, axis=1),
        np.std(chroma, axis=1),
        np.mean(mel, axis=1),
        np.std(mel, axis=1)
    ])

    return features


# ---------- Interface ----------
st.title("🎙️ Speech Emotion Recognition")

st.write(
    "Upload a speech recording and let our AI model "
    "estimate the emotion expressed in the voice."
)

st.info(
    "The model predicts one of eight emotions: "
    "Angry, Calm, Disgust, Fearful, Happy, Neutral, "
    "Sad, or Surprised."
)

uploaded_file = st.file_uploader(
    "Upload an audio recording",
    type=["wav", "mp3", "m4a", "ogg", "flac"]
)

if uploaded_file is not None:
    st.audio(uploaded_file)

    if st.button("🎯 Analyze Emotion", type="primary"):
        temp_path = None

        try:
            # Preserve the uploaded file's extension so
            # librosa can identify the audio format.
            suffix = os.path.splitext(
                uploaded_file.name
            )[1].lower()

            with tempfile.NamedTemporaryFile(
                delete=False,
                suffix=suffix
            ) as temp_file:
                temp_file.write(uploaded_file.getvalue())
                temp_path = temp_file.name

            with st.spinner(
                "Analyzing audio... Please wait!"
            ):
                features = extract_improved_features(
                    temp_path
                )

                if features.shape[0] != model.n_features_in_:
                    st.error(
                        "Feature count mismatch. Expected "
                        f"{model.n_features_in_}, but got "
                        f"{features.shape[0]}."
                    )
                    st.stop()

                features = features.reshape(1, -1)

                prediction = model.predict(features)[0]
                emotion = encoder.inverse_transform(
                    [prediction]
                )[0]

                probabilities = model.predict_proba(
                    features
                )[0]

            # ---------- Results ----------
            st.success("Analysis complete!")

            st.subheader("🎭 Predicted Emotion")
            st.markdown(f"# {emotion.capitalize()}")

            st.subheader("📊 Emotion Confidence Scores")

            sorted_results = sorted(
                zip(encoder.classes_, probabilities),
                key=lambda item: item[1],
                reverse=True
            )

            fig, ax = plt.subplots(figsize=(8, 4))

            labels = [
                label.capitalize()
                for label, probability in sorted_results
            ]
            scores = [
                probability * 100
                for label, probability in sorted_results
            ]

            ax.barh(labels, scores)
            ax.set_xlabel("Model confidence (%)")
            ax.set_xlim(0, 100)
            ax.invert_yaxis()
            fig.tight_layout()

            st.pyplot(fig)
            plt.close(fig)

            for label, probability in sorted_results:
                st.write(
                    f"**{label.capitalize()}** — "
                    f"{probability * 100:.2f}%"
                )

            st.caption(
                "These scores represent the model's estimates, "
                "not guaranteed measurements of a person's "
                "actual emotional state."
            )

        except Exception as e:
            st.error(
                "Could not analyze this audio. Please check "
                "that it is a valid, supported audio file. "
                f"Details: {e}"
            )

        finally:
            if temp_path and os.path.exists(temp_path):
                os.remove(temp_path)


Writing /content/drive/MyDrive/Emotion_Recognition/app.py


In [ ]:

import os

print("Pyngrok installed:", end=" ")
try:
    import pyngrok
    print("Yes")
except ImportError:
    print("No")

print(
    "Ngrok token configured:",
    bool(os.environ.get("NGROK_AUTHTOKEN"))
)


Pyngrok installed: Yes
Ngrok token configured: False


In [ ]:

from getpass import getpass
from pyngrok import ngrok

token = getpass("Paste your ngrok auth token (hidden): ")
ngrok.set_auth_token(token)
del token

print("Ngrok authentication configured for this session!")


Paste your ngrok auth token (hidden): ··········
Ngrok authentication configured for this session!


In [ ]:

from pyngrok import ngrok
import requests

# Remove any old tunnel on port 8501
for tunnel in ngrok.get_tunnels():
    if tunnel.config.get("addr", "").endswith(":8501"):
        ngrok.disconnect(tunnel.public_url)

# Create a new public tunnel
public_url = ngrok.connect(8501, "http").public_url

print("🎉 YOUR SPEECH EMOTION RECOGNITION APP:")
print(public_url)


🎉 YOUR SPEECH EMOTION RECOGNITION APP:
https://livable-skied-feminine.ngrok-free.dev


In [ ]:
!apt-get -qq install -y ffmpeg

In [ ]:

from pyngrok import ngrok
import requests

# Remove any old tunnel on port 8501
for tunnel in ngrok.get_tunnels():
    if tunnel.config.get("addr", "").endswith(":8501"):
        ngrok.disconnect(tunnel.public_url)

# Create a new public tunnel
public_url = ngrok.connect(8501, "http").public_url

print("🎉 YOUR SPEECH EMOTION RECOGNITION APP:")
print(public_url)


🎉 YOUR SPEECH EMOTION RECOGNITION APP:
https://livable-skied-feminine.ngrok-free.dev


In [ ]:
%cd /content/drive/MyDrive/Emotion_Recognition
!ls

[Errno 2] No such file or directory: '/content/drive/MyDrive/Emotion_Recognition'
/content
sample_data


In [ ]:
%cd /content/drive/MyDrive/Emotion_Recognition
!ls

/content/drive/MyDrive/Emotion_Recognition
app.py	Audio_Speech_Actors_01-24.zip  model  processed_data  ravdess


In [ ]:
%cd /content/drive/MyDrive/Emotion_Recognition
!ls


/content/drive/MyDrive/Emotion_Recognition
app_emo_recg.py  Audio_Speech_Actors_01-24.zip	model  processed_data  ravdess


In [ ]:
%cd /content/drive/MyDrive/Emotion_Recognition
!ls -la

/content/drive/MyDrive/Emotion_Recognition
total 203659
drwx------  6 root root      4096 Oct 25 11:44 .
drwx------  9 root root      4096 Oct 25 11:42 ..
-rw-------  1 root root     53001 Oct 25 11:44 app_emo_recg.py
-rw-------  1 root root 208468073 Oct 24 03:31 Audio_Speech_Actors_01-24.zip
drwx------  2 root root      4096 Oct 25 11:44 .ipynb_checkpoints
drwx------  2 root root      4096 Oct 25 08:57 model
drwx------  2 root root      4096 Oct 24 03:45 processed_data
drwx------ 26 root root      4096 Oct 24 03:48 ravdess


In [ ]:
!mkdir -p /content/drive/MyDrive/Emotion_Recognition/.streamlit

In [ ]:
%%writefile /content/drive/MyDrive/Emotion_Recognition/.streamlit/config.toml
[theme]
base = "light"
primaryColor = "#C76543"
backgroundColor = "#FAF7F2"
secondaryBackgroundColor = "#F1EADF"
textColor = "#292724"

[browser]
gatherUsageStats = false

Writing /content/drive/MyDrive/Emotion_Recognition/.streamlit/config.toml


In [ ]:
!pip install -q -U streamlit plotly librosa soundfile pyngrok xgboost scikit-learn
!apt-get -qq install -y ffmpeg

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [ ]:
import streamlit
import librosa
import soundfile
import xgboost
import sklearn
import plotly

print("Streamlit:", streamlit.__version__)
print("Librosa:", librosa.__version__)
print("XGBoost:", xgboost.__version__)
print("Scikit-learn:", sklearn.__version__)
print("All required imports successful!")

Streamlit: 1.66.1
Librosa: 1.0.0
XGBoost: 3.4.1
Scikit-learn: 1.6.1
All required imports successful!


In [ ]:
%cd /content/drive/MyDrive/Emotion_Recognition
!pkill streamlit || true
!nohup streamlit run app_emo_recg.py --server.port 8501 --server.headless true > /content/streamlit.log 2>&1 &

/content/drive/MyDrive/Emotion_Recognition


In [ ]:
!sleep 3
!cat /content/streamlit.log

2026-10-25 11:47:13.070 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.48.176.217:8501



In [ ]:
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Sun, 25 Oct 2026 11:47:52 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 6467
last-modified: Sun, 25 Oct 2026 11:46:25 GMT
etag: "e0953f9e6f336edcdde696b854d29914"
cache-control: no-cache



In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501, "http").public_url
print("OPEN YOUR APP HERE:", public_url)

ERROR:pyngrok.process.ngrok:t=2026-10-25T11:48:12+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-10-25T11:48:12+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
CRITICAL:pyngrok.process.ngrok:t=2026-10-25T11:48:12+0000 lvl=crit msg="command failed" err="authentication failed: This ngrok session is not authenticated. ngrok requi

PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
from pyngrok import ngrok
from getpass import getpass

token = getpass("Paste your ngrok authtoken: ")
ngrok.set_auth_token(token)
print("Ngrok token configured for this runtime!")

Paste your ngrok authtoken: ··········
Ngrok token configured for this runtime!


In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501, "http").public_url
print("🚀 OPEN YOUR APP:", public_url)

🚀 OPEN YOUR APP: https://livable-skied-feminine.ngrok-free.dev


In [ ]:
%%writefile /content/drive/MyDrive/Emotion_Recognition/.streamlit/config.toml
[theme]
base = "light"
primaryColor = "#2F7A58"
backgroundColor = "#F5F8F4"
secondaryBackgroundColor = "#E8EFE7"
textColor = "#1C2620"

[browser]
gatherUsageStats = false

Overwriting /content/drive/MyDrive/Emotion_Recognition/.streamlit/config.toml


In [ ]:
%cd /content/drive/MyDrive/Emotion_Recognition
!pkill streamlit || true
!nohup streamlit run app_emo_recg.py --server.port 8501 --server.headless true > /content/streamlit.log 2>&1 &

/content/drive/MyDrive/Emotion_Recognition


In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501, "http").public_url
print("Your app:", public_url)

Your app: https://livable-skied-feminine.ngrok-free.dev


In [ ]:
%cd /content/drive/MyDrive/Emotion_Recognition
!grep -nE '#(FAF7F2|C76543|E9DFD2|F1EADF|F5F8F4|2F7A58|E8EFE7)' app_emo_recg.py

/content/drive/MyDrive/Emotion_Recognition
66:ACCENT = "#C76543"
150:                fill="#FAF7F2",
185:  --ivory:#FAF7F2; --ivory-2:#F5F0E8; --card:#FFFDFA;
186:  --sand:#E9DFD2; --sand-2:#D9CCBA; --line:#E7DDCF;
188:  --accent:#C76543; --accent-btn:#B8553A; --accent-btn-hover:#A44B32; --accent-ink:#A34A30;
454:        f'<rect x="{8.5 + i * 4.4:.1f}" y="{16 - h / 2:.1f}" width="2.6" height="{h}" rx="1.3" fill="#FAF7F2"/>'
508:        '<g stroke="#FAF7F2" stroke-opacity=".55" stroke-width="2.5" stroke-linecap="round">'
950:        hoverlabel=dict(bgcolor=INK, bordercolor=INK, font=dict(family=FONT_STACK, color="#FAF7F2", size=13)),


In [ ]:
%cd /content/drive/MyDrive/Emotion_Recognition

# Backup your current app
!cp app_emo_recg.py app_emo_recg_backup.py

/content/drive/MyDrive/Emotion_Recognition


In [ ]:
from pathlib import Path

path = Path("/content/drive/MyDrive/Emotion_Recognition/app_emo_recg.py")
code = path.read_text()

replacements = {
    "#C76543": "#2F7A58",
    "#B8553A": "#276749",
    "#A44B32": "#20583F",
    "#A34A30": "#276749",
    "#FAF7F2": "#F5F8F4",
    "#F5F0E8": "#E8EFE7",
    "#FFFDFA": "#FFFFFF",
    "#E9DFD2": "#DCE7DC",
    "#D9CCBA": "#C7D7C7",
    "#E7DDCF": "#D7E2D6",
}

for old, new in replacements.items():
    code = code.replace(old, new)

path.write_text(code)
print("Updated theme colors in app_emo_recg.py")

Updated theme colors in app_emo_recg.py


In [ ]:
!pkill streamlit || true
!nohup streamlit run app_emo_recg.py --server.port 8501 --server.headless true > /content/streamlit.log 2>&1 &

In [ ]:
import glob
import os

dataset_path = "/content/drive/MyDrive/Emotion_Recognition/ravdess"

all_files = glob.glob(
    dataset_path + "/Actor_*/*.wav"
)

angry_files = [
    file for file in all_files
    if os.path.basename(file).split("-")[2] == "05"
]

print("Total angry recordings:", len(angry_files))

for file in angry_files[:10]:
    print(file)

Total angry recordings: 192
/content/drive/MyDrive/Emotion_Recognition/ravdess/Actor_01/03-01-05-01-02-02-01.wav
/content/drive/MyDrive/Emotion_Recognition/ravdess/Actor_01/03-01-05-02-01-02-01.wav
/content/drive/MyDrive/Emotion_Recognition/ravdess/Actor_01/03-01-05-01-02-01-01.wav
/content/drive/MyDrive/Emotion_Recognition/ravdess/Actor_01/03-01-05-02-01-01-01.wav
/content/drive/MyDrive/Emotion_Recognition/ravdess/Actor_01/03-01-05-01-01-01-01.wav
/content/drive/MyDrive/Emotion_Recognition/ravdess/Actor_01/03-01-05-01-01-02-01.wav
/content/drive/MyDrive/Emotion_Recognition/ravdess/Actor_01/03-01-05-02-02-02-01.wav
/content/drive/MyDrive/Emotion_Recognition/ravdess/Actor_01/03-01-05-02-02-01-01.wav
/content/drive/MyDrive/Emotion_Recognition/ravdess/Actor_02/03-01-05-02-01-02-02.wav
/content/drive/MyDrive/Emotion_Recognition/ravdess/Actor_02/03-01-05-02-02-02-02.wav


In [29]:
import os

project_path = "/content/drive/MyDrive/Emotion_Recognition"

for root, dirs, files in os.walk(project_path):
    dirs[:] = [d for d in dirs if d not in [".git", "venv", "__pycache__"]]
    level = root.replace(project_path, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

Emotion_Recognition/
    Audio_Speech_Actors_01-24.zip
    app_emo_recg.py
    app_emo_recg_backup.py
    app.py
    processed_data/
        features.npy
        labels.npy
        emotion_features.csv
        features_improved.npy
        labels_improved.npy
    ravdess/
        Actor_01/
            03-01-03-01-02-01-01.wav
            03-01-05-01-02-02-01.wav
            03-01-05-02-01-02-01.wav
            03-01-08-02-01-01-01.wav
            03-01-07-01-01-02-01.wav
            03-01-02-02-02-01-01.wav
            03-01-01-01-02-01-01.wav
            03-01-06-02-02-01-01.wav
            03-01-05-01-02-01-01.wav
            03-01-05-02-01-01-01.wav
            03-01-06-02-01-02-01.wav
            03-01-04-02-01-02-01.wav
            03-01-03-02-02-01-01.wav
            03-01-05-01-01-01-01.wav
            03-01-03-02-02-02-01.wav
            03-01-01-01-01-01-01.wav
            03-01-07-02-01-02-01.wav
            03-01-06-01-01-01-01.wav
            03-01-06-02-01-01-01.wav
      